In [3]:

import requests, pandas as pd, numpy as np, folium
from folium.plugins import HeatMap
from datetime import datetime, timedelta, timezone
# ---- Map center and Lahore bounding box ----
LAHORE_CENTER = (31.5204, 74.3587)

# ---- Your WAQI (aqicn) token for tile overlay (optional) ----
WAQI_TOKEN = "39c39c83a3be3e9ae0cebbf513917ce76c502e52"  # you shared this

# ---- Optional: UC shapefile path ----
UC_SHP_PATH = "../../data/Lahore UCs/Lahore UC.shp"  # change if stored elsewhere

# bbox as (minLon, minLat, maxLon, maxLat)
BBOX_LONLAT = (74.05, 31.30, 74.60, 31.75)

BP = [
    (0.0, 12.0,   0,  50),
    (12.1, 35.4,  51, 100),
    (35.5, 55.4, 101, 150),
    (55.5,150.4, 151, 200),
    (150.5,250.4,201, 300),
    (250.5,350.4,301, 400),
    (350.5,500.4,401, 500),
]
def pm25_to_aqi(c):
    if c is None or np.isnan(c): return np.nan
    for c_lo, c_hi, i_lo, i_hi in BP:
        if c_lo <= c <= c_hi:
            return round((i_hi - i_lo)/(c_hi - c_lo) * (c - c_lo) + i_lo)
    return 500

def build_heatmap(df_points, add_waqi_tiles=True, waqi_token=WAQI_TOKEN, zoom=11, save_as=None):
    m = folium.Map(location=LAHORE_CENTER, zoom_start=zoom, control_scale=True)
    if add_waqi_tiles and waqi_token:
        folium.raster_layers.TileLayer(
            tiles=f"https://tiles.aqicn.org/tiles/usepa-aqi/{{z}}/{{x}}/{{y}}.png?token={waqi_token}",
            attr="Air Quality tiles © WAQI",
            opacity=0.75,
            name="WAQI EPA AQI tiles",
        ).add_to(m)

    if not df_points.empty:
        heat_data = [[r.latitude, r.longitude, float(r.aqi)] 
                     for r in df_points.itertuples() if not np.isnan(r.aqi)]
        if heat_data:
            HeatMap(heat_data, radius=22, blur=26, max_zoom=12, min_opacity=0.55).add_to(m)
        for r in df_points.itertuples():
            if np.isnan(r.aqi): continue
            folium.CircleMarker(
                [r.latitude, r.longitude], radius=4, weight=1, color="#333",
                fill=True, fill_opacity=0.9,
                popup=f"AQI: {int(r.aqi)}<br>Source: {getattr(r,'source','N/A')}"
            ).add_to(m)
    else:
        folium.Marker(LAHORE_CENTER, tooltip="No points found").add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    if save_as: m.save(save_as)
    m
    return m



In [4]:
minLon, minLat, maxLon, maxLat = BBOX_LONLAT

def fetch_waqi_bounds(token=WAQI_TOKEN, bbox=(minLon, minLat, maxLon, maxLat)):
    url = "https://api.waqi.info/map/bounds/"
    params = {
        "token": token,
        "latlng": f"{bbox[1]},{bbox[0]},{bbox[3]},{bbox[2]}",  # lat1,lon1,lat2,lon2
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    rows = []
    for s in js.get("data", []):
        lat, lon = s.get("lat"), s.get("lon")
        aqi = s.get("aqi")
        # AQI may be "—" or None; skip invalids
        if lat is None or lon is None: 
            continue
        try:
            aqi_val = float(aqi)
        except Exception:
            continue
        rows.append((lat, lon, aqi_val))
    return pd.DataFrame(rows, columns=["latitude","longitude","aqi"]).assign(source="WAQI")

df_waqi = fetch_waqi_bounds()
df_waqi.head(), len(df_waqi)


(Empty DataFrame
 Columns: [latitude, longitude, aqi, source]
 Index: [],
 0)

In [5]:
m_waqi = build_heatmap(df_waqi, add_waqi_tiles=True, waqi_token=WAQI_TOKEN, save_as="lahore_aqi_heatmap_waqi.html")
m_waqi
